# Streaming Products from S3 During Commissioning

## Introduction

This notebook demonstrates how to search MAST for commissioning data products, and then shows how to stream those data products from MAST into memory without the need to make local copies.

This tutorial has been adapted mainly from the MAST notebook [MAST Metadata Search](https://github.com/spacetelescope/mast_notebooks/blob/roman-prelaunch/notebooks/Roman/MAST_metadata_search/MAST_metadata_search.ipynb) and the Nexus tutorial [Working with ADSF](https://github.com/spacetelescope/roman_notebooks/blob/main/notebooks/working_with_asdf/working_with_asdf.ipynb), with additional information added to support streaming from MAST search results.

## Imports

In [1]:
import rosalia as rs
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord  # High-level coordinates

First, we create our `MastMissions` object to act as our gateway to MAST for searches. If you do not have a MAST AUTH token for accessing MAST (needed for Roman OPS access until after commissioning), or if your token has expired, make one now by visiting [MAST.Auth](https://auth.mast.stsci.edu/info).

Once you have a token, you can set it up as an environment variable for later use, or you can store it in a file called `variables.env`. Below, we have assumed the file is in your current working directory (`./`), but the full path to the file can be specified if it is stored elsewhere such as your home directory. The `variables.env` file should look like:

```
MAST_API_TOKEN='yourtokenhere'
```

Next, as an example, here we show how to search for data from Program ID 1039 (CAR-190), pass 1, and detector WFI04. Be as specific as you can as the more results MAST returns the longer this cell will take to execute.

In [ ]:
"""
Example of criteria for a roman_query using astroquery.mast.query_criteria
search = {'program': 1020,
         'observation': 5,
         'pass': 2, 
         #'exposure_start_time': '2026-09-15T02:01:47.4080000',
         'product_type': 'l2', 
         'detector': 'WFI06',
         # 'optical_element': 'F158',
         'exposure_type': "WFI_IMAGE"}
"""

coordinates=SkyCoord(229.64044*u.deg, -3.58747*u.deg, frame="icrs")
radius=10*u.arcsec

# query = roman_query_criteria(search=search, file_suffix="_cal")
results, products = rs.mast.roman_query(coordinates=coordinates, radius=radius, detector="WFI06", file_suffix="_cal")
dm = rs.mast.stream_roman_mast(products=products, row=0)
roman_exposure = rs.core.exposure(dm)

INFO: MAST API token accepted, welcome Alejandro Serrano Borlaff [astroquery.mast.auth]
Mission: roman
Service: search


In [ ]:
roman_exposure.straylight()

In [ ]:
plt.imshow(roman_exposure.DATA[0], vmin=1, vmax=2)

In [ ]:
rs.telescopes.find_filter_in_svo(wavelength="F062",
                                                                                telescope="ROMAN",
                                                                                instrument="WFI",
                                                                                detector="WFI",
                                                                                verbose=True)

In [ ]:
db

## Streaming Files from MAST into Memory

Now that we have search results, we can select one and stream it into memory to examine it. You will primarily be interested in the Level 1 (L1) uncalibrated ramps or the Level 2 (L2) calibrated rate images, but there are some additional products as well. See the [WFI Data Levels and Products](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products) article on RDox for more information.

For the retrieval below, the extension for L1 products is `uncal.asdf`, while for L2 products it is `cal.asdf`.

First, we get the list of data products (files) associated with the search results in the table above. We use the `get_unique_product_list()` method to retrieve only the unique data products.

From the `observation` section of the metadata, we can see that this does conform to our search for our MRT-7b example (we see it is program ID 114, pass 57, etc.). If we want to plot the data, which for MRT-7b is a test pattern, then we can also do that.

<div class="alert alert-warning" style="color:black; background-color:#ffc5c5; border-color:red;">
    <b>NOTE:</b> If you run the cell below, or in any other way try to access part of the file and get a long exception that ends with "ClientResponseError: 403, message='Forbidden'" followed by a long URI, then try to re-run the missions.read_product() command above. We believe this may be due to a timeout during the streaming, and are investigating.
</div>

In [ ]:
fig, ax = plt.subplots()
norm = simple_norm(dm.data, percent=99.9)
ax.imshow(dm.data, origin='lower', norm=norm)
ax.set_title(filtered[tab_index]['dataset'])
ax.set_ylabel('Science Y (pixels)')
ax.set_xlabel('Science X (pixels)');

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12,12))

im = ax.imshow(
    dm.data,
    origin='lower',
    norm=plt.matplotlib.colors.SymLogNorm(linthresh=1.0, vmin=1, vmax=10)
)

ax.set_title(filtered[tab_index]['dataset'])
ax.set_ylabel('Science Y (pixels)')
ax.set_xlabel('Science X (pixels)')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.savefig("first_roman_stray.png", dpi=300) 

In [ ]:
dm.meta

In [ ]:
dm